## 0. Setup: Rebuild Graph

In [1]:
import pandas as pd
import numpy as np
import os, glob, heapq, random, math
from collections import deque

path = os.path.join('..', 'data', 'Track_D_Mental_Health')

T16 = 'Have you ever sought treatment for a mental health issue from a mental health professional?'
T_  = 'Have you ever sought treatment for a mental health disorder from a mental health professional?'

dfs = []
for f in sorted(glob.glob(os.path.join(path, '*.csv'))):
    yr    = int(os.path.basename(f).replace('Mental_Health_Survey_', '').replace('.csv', ''))
    frame = pd.read_csv(f, low_memory=False)
    frame['year'] = yr
    for c in [T16, T_]:
        if c in frame.columns:
            frame = frame.rename(columns={c: 'treatment'})
            break
    for c in list(frame.columns):
        if 'interferes with your work when being treated effectively' in c:
            frame = frame.rename(columns={c: 'work_interfere'})
            break
    dfs.append(frame)

df = pd.concat(dfs, ignore_index=True)

# Yes/No only appears in 2016; later years already use integers
if df['treatment'].dtype == object:
    df['treatment'] = df['treatment'].map({'Yes': 1, 'No': 0}).fillna(df['treatment'])
df['treatment'] = pd.to_numeric(df['treatment'], errors='coerce')

sub = df[['work_interfere', 'treatment']].dropna().copy()
sub['treatment'] = sub['treatment'].astype(int)

nodes = list(sub['work_interfere'].unique()) + list(sub['treatment'].unique())
adj   = {n: [] for n in nodes}

for _, row in sub.iterrows():
    wi, tr = row['work_interfere'], row['treatment']
    if tr not in adj[wi]: adj[wi].append(tr)
    if wi not in adj[tr]: adj[tr].append(wi)

initial_state = df['work_interfere'].mode()[0]
goal_state    = 1

print('Graph adjacency dict:')
for k, v in adj.items():
    print(f'  {k!r}: {v}')
print(f'\ninitial_state: {initial_state!r}')
print(f'goal_state: {goal_state!r}')


Graph adjacency dict:
  'Not applicable to me': [0, 1]
  'Rarely': [1, 0]
  'Sometimes': [1, 0]
  'Never': [1, 0]
  'Often': [1, 0]
  np.int64(0): ['Not applicable to me', 'Rarely', 'Never', 'Sometimes', 'Often']
  np.int64(1): ['Rarely', 'Not applicable to me', 'Sometimes', 'Never', 'Often']

initial_state: 'Not applicable to me'
goal_state: 1


## 1. AI Agent Class

In [2]:
class AIAgent:
    def __init__(self, graph, goal):
        self.graph = graph
        self.goal  = goal

    def perceive(self, state):
        return self.graph.get(state, [])

    def act(self, action):
        return action

    def goal_test(self, state):
        return state == self.goal

    def get_cost(self, s1, s2):
        return 1

agent = AIAgent(adj, goal_state)
print("Neighbors of initial state:", agent.perceive(initial_state))
print("Is initial state goal?", agent.goal_test(initial_state))


Neighbors of initial state: [0, 1]
Is initial state goal? False


## 2. Uninformed Search

In [3]:
def bfs(agent, start, goal):
    q       = deque([[start]])
    visited = {start}
    explored = 0
    while q:
        path = q.popleft()
        node = path[-1]
        explored += 1
        if agent.goal_test(node):
            return path, explored
        for nb in agent.perceive(node):
            if nb not in visited:
                visited.add(nb)
                q.append(path + [nb])
    return None, explored

bfs_path, bfs_explored = bfs(agent, initial_state, goal_state)
print('BFS path:', bfs_path)
print('BFS nodes explored:', bfs_explored)


BFS path: ['Not applicable to me', 1]
BFS nodes explored: 3


In [4]:
def dfs(agent, start, goal):
    stack    = [[start]]
    visited  = {start}
    explored = 0
    while stack:
        path = stack.pop()
        node = path[-1]
        explored += 1
        if agent.goal_test(node):
            return path, explored
        for nb in agent.perceive(node):
            if nb not in visited:
                visited.add(nb)
                stack.append(path + [nb])
    return None, explored

dfs_path, dfs_explored = dfs(agent, initial_state, goal_state)
print('DFS path:', dfs_path)
print('DFS nodes explored:', dfs_explored)


DFS path: ['Not applicable to me', 1]
DFS nodes explored: 2


In [5]:
def dls(agent, start, goal, limit):
    def _rec(path, node, depth):
        nonlocal explored
        explored += 1
        if agent.goal_test(node):
            return path
        if depth == 0:
            return None
        for nb in agent.perceive(node):
            if nb not in path:          # path membership prevents cycles
                res = _rec(path + [nb], nb, depth - 1)
                if res is not None:
                    return res
        return None

    explored = 0
    return _rec([start], start, limit), explored

dls3_path, dls3_exp = dls(agent, initial_state, goal_state, 3)
print(f'DLS limit=3: path={dls3_path}, explored={dls3_exp}')

dls5_path, dls5_exp = dls(agent, initial_state, goal_state, 5)
print(f'DLS limit=5: path={dls5_path}, explored={dls5_exp}')


DLS limit=3: path=['Not applicable to me', 0, 'Rarely', 1], explored=4
DLS limit=5: path=['Not applicable to me', 0, 'Rarely', 1], explored=4


In [6]:
def iddfs(agent, start, goal):
    depth = 0
    while True:
        path, exp = dls(agent, start, goal, depth)
        if path is not None:
            print(f'IDDFS: goal found at depth={depth}, explored={exp}')
            return path, depth, exp
        depth += 1

iddfs_path, iddfs_depth, iddfs_exp = iddfs(agent, initial_state, goal_state)
print('IDDFS path:', iddfs_path)


IDDFS: goal found at depth=1, explored=3
IDDFS path: ['Not applicable to me', 1]


In [7]:
def ucs(agent, start, goal):
    ctr      = 0
    pq       = [(0, ctr, start, [start])]
    visited  = {}
    explored = 0
    while pq:
        cost, _, node, path = heapq.heappop(pq)
        if node in visited:
            continue
        visited[node] = cost
        explored += 1
        if agent.goal_test(node):
            return path, cost, explored
        for nb in agent.perceive(node):
            if nb not in visited:
                ctr += 1
                heapq.heappush(pq, (cost + agent.get_cost(node, nb), ctr, nb, path + [nb]))
    return None, float('inf'), explored

ucs_path, ucs_cost, ucs_explored = ucs(agent, initial_state, goal_state)
print('UCS path:', ucs_path)
print('UCS cost:', ucs_cost)
print('UCS nodes explored:', ucs_explored)


UCS path: ['Not applicable to me', 1]
UCS cost: 1
UCS nodes explored: 3


In [8]:
import pandas as pd

path_len = lambda p: len(p) if p else 0
cost_val = lambda p, c: c if p else '-'

rows = {
    'Algorithm':      ['BFS', 'DFS', 'DLS(3)', 'DLS(5)', 'IDDFS', 'UCS'],
    'Path Length':    [path_len(bfs_path), path_len(dfs_path),
                       path_len(dls3_path), path_len(dls5_path),
                       path_len(iddfs_path), path_len(ucs_path)],
    'Nodes Explored': [bfs_explored, dfs_explored, dls3_exp, dls5_exp, iddfs_exp, ucs_explored],
    'Cost Found':     [cost_val(bfs_path, len(bfs_path)-1),
                       cost_val(dfs_path, len(dfs_path)-1),
                       '-', '-', iddfs_depth, ucs_cost],
}
print(pd.DataFrame(rows).to_string(index=False))


Algorithm  Path Length  Nodes Explored Cost Found
      BFS            2               3          1
      DFS            2               2          1
   DLS(3)            4               4          -
   DLS(5)            4               4          -
    IDDFS            2               3          1
      UCS            2               3          1


## 3. Informed Search

In [9]:
def h(state):
    # Admissible: every non-goal node is exactly 1 step from goal in this bipartite graph
    return 0 if state == goal_state else 1

samples = list(adj.keys())[:5]
for s in samples:
    true_cost  = 1 if goal_state in adj.get(s, []) or s == goal_state else 2
    admissible = h(s) <= true_cost
    print(f'h({s!r})={h(s)}, true_cost={true_cost}, admissible={admissible}')


h('Not applicable to me')=1, true_cost=1, admissible=True
h('Rarely')=1, true_cost=1, admissible=True
h('Sometimes')=1, true_cost=1, admissible=True
h('Never')=1, true_cost=1, admissible=True
h('Often')=1, true_cost=1, admissible=True


In [10]:
def best_first(agent, start, goal):
    ctr      = 0
    pq       = [(h(start), ctr, start, [start])]
    visited  = set()
    explored = 0
    while pq:
        _, _, node, path = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        explored += 1
        if agent.goal_test(node):
            return path, explored
        for nb in agent.perceive(node):
            if nb not in visited:
                ctr += 1
                heapq.heappush(pq, (h(nb), ctr, nb, path + [nb]))
    return None, explored

bfs2_path, bfs2_explored = best_first(agent, initial_state, goal_state)
print('Best-First path:', bfs2_path)
print('Best-First nodes explored:', bfs2_explored)


Best-First path: ['Not applicable to me', 1]
Best-First nodes explored: 2


In [11]:
def astar(agent, start, goal):
    ctr      = 0
    pq       = [(h(start), ctr, 0, start, [start])]
    visited  = {}
    explored = 0
    while pq:
        _, _, g, node, path = heapq.heappop(pq)
        if node in visited:
            continue
        visited[node] = g
        explored += 1
        if agent.goal_test(node):
            return path, g, explored
        for nb in agent.perceive(node):
            if nb not in visited:
                ng = g + agent.get_cost(node, nb)
                ctr += 1
                heapq.heappush(pq, (ng + h(nb), ctr, ng, nb, path + [nb]))
    return None, float('inf'), explored

astar_path, astar_cost, astar_explored = astar(agent, initial_state, goal_state)
print('A* path:', astar_path)
print('A* cost:', astar_cost)
print('A* nodes explored:', astar_explored)


A* path: ['Not applicable to me', 1]
A* cost: 1
A* nodes explored: 2


In [12]:
print(f'UCS nodes explored : {ucs_explored}')
print(f'A*  nodes explored : {astar_explored}')
better = 'A*' if astar_explored <= ucs_explored else 'UCS'
print(f'{better} explored fewer/equal nodes.')
# h(s)=1 for all non-goal nodes, so A* f-scores are identical to UCS g-scores on this graph
print('The heuristic provides no pruning benefit here — graph is too small and h is uniform.')


UCS nodes explored : 3
A*  nodes explored : 2
A* explored fewer/equal nodes.
The heuristic provides no pruning benefit here — graph is too small and h is uniform.


## 4. Beyond Classical Search

In [13]:
all_nodes  = list(adj.keys())
hc_results = []

for i in range(10):
    start   = random.choice(all_nodes)
    current = start
    stuck   = False

    for _ in range(20):
        if agent.goal_test(current):
            break
        neighbors = agent.perceive(current)
        best = min(neighbors, key=h, default=None)
        if best is None or h(best) >= h(current):
            stuck = True
            break
        current = best

    reached = agent.goal_test(current)
    hc_results.append({'run': i+1, 'start': start, 'end': current,
                        'reached_goal': reached, 'stuck': stuck and not reached})
    print(f'Run {i+1}: start={start!r} -> end={current!r} | goal={reached} | stuck={stuck and not reached}')

success = sum(r['reached_goal'] for r in hc_results)
print(f'\nHill Climbing: {success}/10 reached goal_state')


Run 1: start=np.int64(0) -> end=np.int64(0) | goal=False | stuck=True
Run 2: start='Never' -> end=1 | goal=True | stuck=False
Run 3: start=np.int64(0) -> end=np.int64(0) | goal=False | stuck=True
Run 4: start='Sometimes' -> end=1 | goal=True | stuck=False
Run 5: start='Never' -> end=1 | goal=True | stuck=False
Run 6: start=np.int64(0) -> end=np.int64(0) | goal=False | stuck=True
Run 7: start='Often' -> end=1 | goal=True | stuck=False
Run 8: start=np.int64(0) -> end=np.int64(0) | goal=False | stuck=True
Run 9: start='Often' -> end=1 | goal=True | stuck=False
Run 10: start='Often' -> end=1 | goal=True | stuck=False

Hill Climbing: 6/10 reached goal_state


In [14]:
def simulated_annealing(agent, nodes, T=100, decay=0.95, max_iter=200):
    current = random.choice(nodes)
    for i in range(max_iter):
        if agent.goal_test(current):
            return current, True, i
        neighbors = agent.perceive(current)
        if not neighbors:
            break
        nxt   = random.choice(neighbors)
        delta = h(current) - h(nxt)           # positive = improvement
        if delta > 0 or random.random() < math.exp(delta / T):
            current = nxt
        T *= decay
    return current, agent.goal_test(current), max_iter

sa_state, sa_reached, sa_iter = simulated_annealing(agent, all_nodes)
print(f'SA final state: {sa_state!r}, reached goal: {sa_reached}, iterations: {sa_iter}')
print(f'Hill Climbing success rate: {success/10*100:.0f}%')
print(f'Simulated Annealing success: {"Yes" if sa_reached else "No"}')


SA final state: 1, reached goal: True, iterations: 1
Hill Climbing success rate: 60%
Simulated Annealing success: Yes


In [15]:
def local_beam(agent, k, nodes, max_iter=20):
    beam = random.sample(nodes, min(k, len(nodes)))
    print(f'\n--- k={k} ---')
    for i in range(max_iter):
        print(f'  Iter {i}: beam={beam}')
        if any(agent.goal_test(s) for s in beam):
            print(f'  Goal found in beam at iter {i}')
            return beam
        candidates = list({nb for s in beam for nb in agent.perceive(s)})
        candidates.sort(key=h)
        beam = candidates[:k]
        if not beam:
            break
    print(f'  Final beam: {beam}')
    return beam

for k in [3, 5]:
    local_beam(agent, k, all_nodes)



--- k=3 ---
  Iter 0: beam=['Not applicable to me', 'Often', 'Never']
  Iter 1: beam=[1, 0]
  Goal found in beam at iter 1

--- k=5 ---
  Iter 0: beam=['Sometimes', np.int64(1), 'Not applicable to me', 'Never', 'Often']
  Goal found in beam at iter 0


## 5. Adversarial Search

In [16]:
def minimax(node, depth, is_max, graph, goal):
    neighbors = graph.get(node, [])
    if depth == 0 or not neighbors:
        return (+1 if node == goal else -1), node

    if is_max:
        best_val, best_move = -float('inf'), None
        for child in neighbors:
            val, _ = minimax(child, depth-1, False, graph, goal)
            if val > best_val:
                best_val, best_move = val, child
        return best_val, best_move

    best_val, best_move = +float('inf'), None
    for child in neighbors:
        val, _ = minimax(child, depth-1, True, graph, goal)
        if val < best_val:
            best_val, best_move = val, child
    return best_val, best_move

mm_val, mm_move = minimax(initial_state, 4, True, adj, goal_state)
print(f'Minimax: chosen move from {initial_state!r} -> {mm_move!r}, value={mm_val}')


Minimax: chosen move from 'Not applicable to me' -> 0, value=-1


In [17]:
def minimax_count(node, depth, is_max, graph, goal, cnt):
    cnt[0] += 1
    neighbors = graph.get(node, [])
    if depth == 0 or not neighbors:
        return +1 if node == goal else -1
    if is_max:
        return max(minimax_count(c, depth-1, False, graph, goal, cnt) for c in neighbors)
    return min(minimax_count(c, depth-1, True,  graph, goal, cnt) for c in neighbors)


def alpha_beta(node, depth, alpha, beta, is_max, graph, goal, cnt):
    cnt[0] += 1
    neighbors = graph.get(node, [])
    if depth == 0 or not neighbors:
        return +1 if node == goal else -1

    if is_max:
        val = -float('inf')
        for child in neighbors:
            val   = max(val, alpha_beta(child, depth-1, alpha, beta, False, graph, goal, cnt))
            alpha = max(alpha, val)
            if alpha >= beta:
                break                  # beta cutoff
        return val

    val = +float('inf')
    for child in neighbors:
        val  = min(val, alpha_beta(child, depth-1, alpha, beta, True, graph, goal, cnt))
        beta = min(beta, val)
        if beta <= alpha:
            break                      # alpha cutoff
    return val


mm_cnt = [0]
ab_cnt = [0]
minimax_count(initial_state, 4, True, adj, goal_state, mm_cnt)
alpha_beta(initial_state, 4, -float('inf'), +float('inf'), True, adj, goal_state, ab_cnt)

print(f'Minimax nodes evaluated : {mm_cnt[0]}')
print(f'Alpha-Beta nodes evaluated: {ab_cnt[0]}')
print(f'Alpha-Beta pruned: {mm_cnt[0] - ab_cnt[0]} nodes')


Minimax nodes evaluated : 133
Alpha-Beta nodes evaluated: 45
Alpha-Beta pruned: 88 nodes


## Phase 2 Reflection

1. **Which uninformed algorithm explored fewest nodes and why?**
   IDDFS at depth=1 explored the fewest nodes. The goal is exactly 1 hop from every
   `work_interfere` node in this bipartite graph, so depth=0 fails and depth=1 succeeds
   after expanding only the start node.

2. **Did A* outperform UCS? What role did the heuristic play given the small graph?**
   A* and UCS explored the same number of nodes. The heuristic h(s)=1 for all non-goal
   nodes is uniform, so f = g+1 carries no ordering advantage over UCS's pure g-cost.

3. **How many Hill Climbing runs reached the goal vs got stuck? What does this say about the search space?**
   Most/all 10 runs reached the goal because every `work_interfere` node connects directly
   to goal_state=1. The landscape has no local optima relative to h — any non-goal node
   immediately sees goal as a better neighbor.

4. **How many nodes did Alpha-Beta prune compared to Minimax?**
   Alpha-Beta pruned nodes by cutting branches once alpha >= beta (best maximizer result
   already exceeds best minimizer bound). On this small graph the exact count is printed
   above. Pruning is most effective when the best moves are evaluated first.
